# Lecture 11 - Model integration (Part 1)

Assignment: Serialization and ONNX export

Instructions:

Train a small model
Save and load it
Export to ONNX and compare outputs

Task 1: Save and load
Train a small model and verify persistence.

In [2]:
# TODO: Load it back and verify predictions

from pathlib import Path

import numpy as np


# Load the L11 ResNet18-CIFAR10 model (saved from L11 notebook) and optionally run via ONNX.
# We're doing some pathing to fetch the models from the neighboring folder.
# L11_DIR = Path(__file__).resolve().parent.parent / "L11"

# Notebook directory as base
# The line of code above works fine in a .py file, but __file__ is not defined
# because we are writing code inside cells.
L11_DIR = Path(".")
print(L11_DIR)
L11_PT = L11_DIR / "L11_model.pt"
L11_ONNX = L11_DIR / "L11_model.onnx"


if L11_PT.exists():
    import torch
    from torch.nn import Linear
    from torchvision import models

    _l11_model = models.resnet18(weights=None)
    _l11_model.fc = Linear(_l11_model.fc.in_features, 10)
    _l11_model.load_state_dict(torch.load(L11_PT, map_location="cpu", weights_only=True))
    _l11_model.eval()
    # Quick sanity check: one batch shape (1, 3, 32, 32) -> (1, 10)
    _x = torch.randn(1, 3, 32, 32)
    with torch.no_grad():
        _out = _l11_model(_x)
    print("L11 model loaded from L11_model.pt; output shape:", _out.shape)

.
L11 model loaded from L11_model.pt; output shape: torch.Size([1, 10])


### Task 2: ONNX export
Export and compare outputs if ONNX tools are available.

In [3]:
# TODO: Run a test inference with onnxruntime (if available)

if L11_ONNX.exists():
    try:
        import onnxruntime as ort
        _sess = ort.InferenceSession(str(L11_ONNX))
        _name = _sess.get_inputs()[0].name
        _dummy = np.random.randn(1, 3, 32, 32).astype(np.float32)
        _onnx_out = _sess.run(None, {_name: _dummy})[0]
        print("L11 ONNX run output shape:", _onnx_out.shape)
    except ImportError:
        pass
if not L11_PT.exists() and not L11_ONNX.exists():
    print("L11 model files not found (run L11 notebook to create L11_model.pt and L11_model.onnx).")

L11 ONNX run output shape: (1, 10)


In [ ]:
# TODO: Compare outputs with the original model

# Load in test data (CIFAR10) and run inference with both models to compare outputs
from torchvision import datasets, transforms    

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])  

testset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4, shuffle=False)
dataiter = iter(testloader)

# Output accuracy and loss for both models on the same batch, and compare outputs
with torch.no_grad():
    for images, labels in testloader:

        original_out = _l11_model(images)
        print("Original model output shape:", original_out.shape)
    
        if L11_ONNX.exists():
            onnx_out = _sess.run(None, {_name: images.numpy()})[0]
            print("ONNX model output shape:", onnx_out.shape)   
    